# 03 - Test Evaluation
Person 3. Loads model + test.csv, reports performance metrics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
sys.path.append('../src')
from utils import TARGET

test_df = pd.read_csv('../data/test.csv')
model = joblib.load('../model/attrition_model.pkl')
model_columns = joblib.load('../model/model_columns.pkl')
test_df.head()


## Prep test set to match training columns

In [ ]:
df = test_df.copy()
df[TARGET] = df[TARGET].map({'Yes': 1, 'No': 0})

cat_cols = df.select_dtypes(include='object').columns
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# align columns with training set (adds missing cols as 0, drops extras)
df = df.reindex(columns=model_columns + [TARGET], fill_value=0)

X_test = df[model_columns]
y_test = df[TARGET]


## Predict & score

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print('Accuracy:', accuracy_score(y_test, y_pred))
print('ROC AUC:', roc_auc_score(y_test, y_prob))
print()
print(classification_report(y_test, y_pred, target_names=['No', 'Yes']))


## Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Test Set')
plt.savefig('../outputs/charts/confusion_matrix.png', bbox_inches='tight')
plt.show()


## ROC curve

In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure()
plt.plot(fpr, tpr, label=f'AUC = {roc_auc_score(y_test, y_prob):.2f}')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Test Set')
plt.legend()
plt.savefig('../outputs/charts/roc_curve.png', bbox_inches='tight')
plt.show()
